In [58]:

# in this notebook we will implement the remaining parts of the gpt model 
# we have already implemented the multihead attention mechanism
# we will need now to implement the feed forward network the layer normalization and the residual connections
# then finally stack them all together to form the transformer block
# we will also implement the GELU function (gaussian error linear unit) which is a smooth approximation of the ReLU function
# and is used as the activation function in the feed forward network
# its advantage is that it is differentiable and has a non-zero gradient for negative inputs 
# which helps with the vanishing gradient problem
# it tackles the dying relu problem by allowing negative inputs to have a small positive output
# except at approximately -0.75 


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import torch
from torch import nn

torch.manual_seed(42)

In [60]:
# a layer normalization layer is a type of normalization layer that normalizes the inputs before passing them the next layer
# this is done by subtracting the mean and dividing by the standard deviation of the inputs
# however we also multiply by learnable scale and add a learnable shift parameter to the normalized inputs 
# which will allow the model to learn optimal scale and shift if that is what will make the model perform better
# provides some kind of flexibility to the model to learn the optimal scale and shift parameters for the inputs

class LayerNorm(nn.Module):
    def __init__(self, din):
        super().__init__()

        self.gamma = nn.Parameter(torch.ones(din))
        self.beta = nn.Parameter(torch.zeros(din))

    def forward(self, X):
        mean = X.mean(dim = -1, keepdim = True)
        var = X.var(dim = -1, keepdim = True, unbiased = False)  # this prevents the bias towards the /n or /n-1 (bessels correction)
        std = torch.sqrt(var + 1e-5)  # add a small value to prevent division by zero

        norm_x = (X - mean) / std

        return self.gamma * norm_x + self.beta